# ⚙️ Credit Risk & Loan Default Prediction: 02 - Feature Engineering & Preprocessing

### Objective:
Construct robust custom transformers for missing value indicators, financial ratios (Credit/Income, Annuity/Income, Goods/Income), age & employment anomaly transformations, and column pruning.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin

pd.set_option('display.max_columns', 100)


## 1. Train-Test Split (Stratified)


In [ ]:
df = pd.read_csv('../data/raw/application_train.csv')
X = df.drop(columns='TARGET')
y = df['TARGET']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'X_train shape: {X_train.shape}, X_test shape: {X_test.shape}')


## 2. Identify Categorical & Numeric Categorical Features


In [ ]:
numeric_cols = X_train.select_dtypes(include='number')
categorical_numeric = numeric_cols.columns[numeric_cols.nunique() < 63].tolist()

cols_to_remove = [
    'TARGET', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'HOUR_APPR_PROCESS_START',
    'ELEVATORS_MODE', 'ENTRANCES_MODE', 'FLOORSMAX_MODE', 'FLOORSMIN_MODE',
    'ELEVATORS_MEDI', 'ENTRANCES_MEDI', 'FLOORSMAX_MEDI', 'FLOORSMIN_MEDI',
    'OBS_30_CNT_SOCIAL_CIRCLE', 'DEF_30_CNT_SOCIAL_CIRCLE',
    'OBS_60_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE', 'OWN_CAR_AGE'
]

categorical_numeric = [col for col in categorical_numeric if col not in cols_to_remove]
print(f'Categorical Numeric columns: {len(categorical_numeric)}')


## 3. High-Dimensional / Redundant Columns to Drop


In [ ]:
coltodrop = [
    'SK_ID_CURR', 'FLAG_CONT_MOBILE', 'FLAG_EMAIL', 'REG_REGION_NOT_LIVE_REGION',
    'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'YEARS_BEGINEXPLUATATION_AVG',
    'COMMONAREA_AVG', 'ENTRANCES_AVG', 'LANDAREA_AVG', 'NONLIVINGAPARTMENTS_AVG',
    'NONLIVINGAREA_AVG', 'BASEMENTAREA_MODE', 'YEARS_BEGINEXPLUATATION_MODE',
    'COMMONAREA_MODE', 'ENTRANCES_MODE', 'LANDAREA_MODE', 'NONLIVINGAPARTMENTS_MODE',
    'NONLIVINGAREA_MODE', 'YEARS_BEGINEXPLUATATION_MEDI', 'COMMONAREA_MEDI',
    'ENTRANCES_MEDI', 'LANDAREA_MEDI', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_MEDI',
    'OBS_30_CNT_SOCIAL_CIRCLE', 'OBS_60_CNT_SOCIAL_CIRCLE', 'FLAG_DOCUMENT_2',
    'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_8',
    'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12',
    'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16',
    'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20',
    'FLAG_DOCUMENT_21', 'AMT_REQ_CREDIT_BUREAU_HOUR', 'AMT_REQ_CREDIT_BUREAU_DAY',
    'AMT_REQ_CREDIT_BUREAU_WEEK', 'AMT_REQ_CREDIT_BUREAU_MON', 'AMT_REQ_CREDIT_BUREAU_QRT',
    'AMT_REQ_CREDIT_BUREAU_YEAR'
]
print(f'Total columns designated to drop: {len(coltodrop)}')


## 4. Custom Feature Engineering Transformer


In [ ]:
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, cols_to_drop=None, categorical_numeric=None):
        self.cols_to_drop = cols_to_drop or []
        self.categorical_numeric = categorical_numeric or []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Row-level missingness indicators
        X['TOTAL_MISSING'] = X.isna().sum(axis=1)
        X['MISSING_RATIO'] = X.isna().mean(axis=1)

        # Categorical numeric encoding as strings
        for col in self.categorical_numeric:
            if col in X.columns:
                X[col] = X[col].apply(
                    lambda value: f'UNKNOWN_{col}' if pd.isna(value) else f'{col}_{value}'
                )

        # Age in Years
        if 'DAYS_BIRTH' in X.columns:
            X['AGE_YEARS'] = -X['DAYS_BIRTH'] / 365.25

        # Employment Anomaly & Years
        if 'DAYS_EMPLOYED' in X.columns:
            X['EMPLOYED_ANOMALY'] = (X['DAYS_EMPLOYED'] > 0).astype(int)
            X['EMPLOYED_YEARS'] = (
                X['DAYS_EMPLOYED'].where(X['DAYS_EMPLOYED'] <= 0) / -365.25
            )

        # Financial ratios
        if 'AMT_CREDIT' in X.columns and 'AMT_INCOME_TOTAL' in X.columns:
            X['CREDIT_INCOME_RATIO'] = X['AMT_CREDIT'] / X['AMT_INCOME_TOTAL'].replace(0, np.nan)
        if 'AMT_ANNUITY' in X.columns and 'AMT_INCOME_TOTAL' in X.columns:
            X['ANNUITY_INCOME_RATIO'] = X['AMT_ANNUITY'] / X['AMT_INCOME_TOTAL'].replace(0, np.nan)
        if 'AMT_GOODS_PRICE' in X.columns and 'AMT_INCOME_TOTAL' in X.columns:
            X['GOODS_INCOME_RATIO'] = X['AMT_GOODS_PRICE'] / X['AMT_INCOME_TOTAL'].replace(0, np.nan)
        if 'AMT_INCOME_TOTAL' in X.columns and 'CNT_FAM_MEMBERS' in X.columns:
            X['INCOME_PER_FAMILY_MEMBER'] = X['AMT_INCOME_TOTAL'] / X['CNT_FAM_MEMBERS'].replace(0, np.nan)

        # Car Age Missing
        if 'OWN_CAR_AGE' in X.columns:
            X['OWN_CAR_AGE_MISSING'] = X['OWN_CAR_AGE'].isna().astype(int)
            X['OWN_CAR_AGE'] = X['OWN_CAR_AGE'].fillna(0)

        # Drop selected columns
        X = X.drop(columns=self.cols_to_drop, errors='ignore')
        return X

fe = FeatureEngineer(cols_to_drop=coltodrop, categorical_numeric=categorical_numeric)
X_train_fe = fe.fit_transform(X_train)
print(f'Transformed shape: {X_train_fe.shape}')
X_train_fe.head(3)
